
# NurtureJoy Notebook 1 — Merge Datasets + EDA

This notebook:
1. loads all source datasets,
2. standardizes them into a common schema,
3. creates separate **emotion** and **safety** data pools,
4. runs EDA and exports merged CSV files for the next notebooks.

**Expected input files**
- `Combined_Data.csv`
- `goemotions_1.csv`
- `goemotions_2.csv`
- `goemotions_3.csv`
- `Suicide_Detection.csv`
- `train.csv` (Jigsaw toxic comments)

This notebook is designed to run in **Google Colab** or **local Jupyter / VS Code**.


In [ ]:

# If running in Colab, uncomment:
# !pip -q install pandas numpy matplotlib seaborn scikit-learn

import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 180)
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:

# ----------------------------
# Path setup
# ----------------------------
BASE_DIR = Path(".")  # Colab or local current folder

PATHS = {
    "combined": BASE_DIR / "Combined_Data.csv",
    "go1": BASE_DIR / "goemotions_1.csv",
    "go2": BASE_DIR / "goemotions_2.csv",
    "go3": BASE_DIR / "goemotions_3.csv",
    "suicide": BASE_DIR / "Suicide_Detection.csv",
    "jigsaw": BASE_DIR / "train.csv",
}

missing = [str(p) for p in PATHS.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

PATHS


In [ ]:

def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def word_count(text):
    return len(str(text).split())

def char_count(text):
    return len(str(text))



## 1. Load and standardize each source
We create a common schema:

- `text`
- `source`
- `raw_label`
- `source_group`
- `use_for_emotion`
- `use_for_safety`


In [ ]:

# Combined_Data.csv
combined = pd.read_csv(PATHS["combined"])
combined = combined.rename(columns={"statement": "text", "status": "raw_label"})
combined = combined[["text", "raw_label"]].copy()
combined["source"] = "Combined_Data"
combined["source_group"] = "maternal_mental_health"
combined["use_for_emotion"] = True
combined["use_for_safety"] = True
combined["text"] = combined["text"].map(clean_text)

combined.head()


In [ ]:

# GoEmotions
go1 = pd.read_csv(PATHS["go1"])
go2 = pd.read_csv(PATHS["go2"])
go3 = pd.read_csv(PATHS["go3"])
go = pd.concat([go1, go2, go3], ignore_index=True)

emotion_cols = [
    c for c in go.columns
    if c not in ["text", "id", "author", "subreddit", "link_id", "parent_id", "created_utc", "rater_id", "example_very_unclear"]
]

# Keep rows that are not marked unclear
go = go[go["example_very_unclear"] == False].copy()

def get_primary_goemotion(row):
    active = [c for c in emotion_cols if row[c] == 1]
    if not active:
        return None
    return active[0]  # deterministic primary label

go["raw_label"] = go.apply(get_primary_goemotion, axis=1)
go = go[["text", "raw_label"]].copy()
go["source"] = "GoEmotions"
go["source_group"] = "general_emotions"
go["use_for_emotion"] = True
go["use_for_safety"] = False
go["text"] = go["text"].map(clean_text)

go.head()


In [ ]:

# Suicide_Detection.csv
suicide = pd.read_csv(PATHS["suicide"])
suicide = suicide.rename(columns={"class": "raw_label"})
suicide = suicide[["text", "raw_label"]].copy()
suicide["source"] = "Suicide_Detection"
suicide["source_group"] = "high_distress"
suicide["use_for_emotion"] = True
suicide["use_for_safety"] = True
suicide["text"] = suicide["text"].map(clean_text)

suicide.head()


In [ ]:

# Jigsaw Toxic Comments
jigsaw = pd.read_csv(PATHS["jigsaw"])
tox_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

jigsaw["raw_label"] = np.where(jigsaw[tox_cols].max(axis=1) > 0, "unsafe_toxic", "safe_non_toxic")
jigsaw = jigsaw.rename(columns={"comment_text": "text"})
jigsaw = jigsaw[["text", "raw_label"]].copy()
jigsaw["source"] = "Jigsaw_Toxic"
jigsaw["source_group"] = "toxicity"
jigsaw["use_for_emotion"] = False   # reserve for safety only
jigsaw["use_for_safety"] = True
jigsaw["text"] = jigsaw["text"].map(clean_text)

jigsaw.head()


In [ ]:

# Master merged dataset
master = pd.concat([combined, go, suicide, jigsaw], ignore_index=True)
master = master.dropna(subset=["text"])
master["word_count"] = master["text"].map(word_count)
master["char_count"] = master["text"].map(char_count)
master["is_duplicate_text"] = master.duplicated(subset=["text"], keep=False)

print(master.shape)
master.head()



## 2. Basic EDA


In [ ]:

print("Rows by source:")
display(master["source"].value_counts().to_frame("rows"))

print("Rows by source_group:")
display(master["source_group"].value_counts().to_frame("rows"))

print("Raw labels per source (top 20 each):")
for src, sub in master.groupby("source"):
    print("\nSOURCE:", src)
    display(sub["raw_label"].value_counts().head(20).to_frame("count"))


In [ ]:

print("Null text rows:", master["text"].isna().sum())
print("Duplicate text rows:", master["is_duplicate_text"].sum())

display(master[["word_count", "char_count"]].describe())


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
master["word_count"].clip(upper=200).hist(ax=ax[0], bins=50)
ax[0].set_title("Word Count Distribution (clipped at 200)")
ax[0].set_xlabel("Words")

master["char_count"].clip(upper=1000).hist(ax=ax[1], bins=50)
ax[1].set_title("Character Count Distribution (clipped at 1000)")
ax[1].set_xlabel("Characters")
plt.show()


In [ ]:

# Very short / suspicious rows
short_rows = master[master["word_count"] <= 3].copy()
print("Very short rows:", len(short_rows))
display(short_rows.sample(min(20, len(short_rows)), random_state=42))



## 3. Create the emotion and safety pools
We keep:
- **emotion pool** = Combined_Data + GoEmotions + Suicide_Detection
- **safety pool** = Combined_Data + Suicide_Detection + Jigsaw

Jigsaw is kept out of the emotion labeling pool because toxic forum comments are helpful for safety training, but they can contaminate the emotional support labels for a pregnancy chatbot.


In [ ]:

emotion_pool = master[master["use_for_emotion"]].copy().reset_index(drop=True)
safety_pool = master[master["use_for_safety"]].copy().reset_index(drop=True)

print("Emotion pool shape:", emotion_pool.shape)
print("Safety pool shape:", safety_pool.shape)


In [ ]:

# Source-prior mapping coverage preview
source_map_preview = {
    # Combined_Data
    "Anxiety": "ANXIETY",
    "Stress": "STRESS",
    "Depression": "LOW_MOOD",
    "Suicidal": "HIGH_DISTRESS",
    "Normal": "NEUTRAL",
    "Bipolar": "LOW_MOOD",
    "Personality disorder": "LOW_MOOD",

    # Suicide_Detection
    "suicide": "HIGH_DISTRESS",
    "non-suicide": None,  # leave for later / zero-shot / relabeling

    # Jigsaw
    "unsafe_toxic": None,
    "safe_non_toxic": None,

    # GoEmotions direct map examples
    "joy": "POSITIVE",
    "gratitude": "POSITIVE",
    "love": "POSITIVE",
    "optimism": "POSITIVE",
    "relief": "POSITIVE",
    "neutral": "NEUTRAL",
    "fear": "ANXIETY",
    "nervousness": "ANXIETY",
    "anger": "STRESS",
    "annoyance": "STRESS",
    "sadness": "LOW_MOOD",
    "grief": "LOW_MOOD",
    "disappointment": "LOW_MOOD",
    "remorse": "LOW_MOOD",
}

emotion_pool["prior_label"] = emotion_pool["raw_label"].map(source_map_preview)
display(emotion_pool["prior_label"].value_counts(dropna=False).to_frame("count"))


In [ ]:

# Export for next notebooks
master.to_csv("nurturejoy_master_merged_raw.csv", index=False)
emotion_pool.to_csv("nurturejoy_emotion_pool_raw.csv", index=False)
safety_pool.to_csv("nurturejoy_safety_pool_raw.csv", index=False)

print("Saved:")
print("- nurturejoy_master_merged_raw.csv")
print("- nurturejoy_emotion_pool_raw.csv")
print("- nurturejoy_safety_pool_raw.csv")
